# SSID 9-class - Augmentation A/B experiment

Trains **two YOLO11n detectors on the same data with the same hyperparameters**. The only thing that
differs is data augmentation: **Run A = OFF**, **Run B = ON**. Then it builds the full figure + table
report and compares them.

**Why this experiment:** `needle` and the other small instruments have been the weak classes.
Ultralytics' default augmentation (mosaic, HSV, scale, translate, horizontal flip) usually helps
overall on a small training set, but mosaic in particular can *hurt* small objects. The per-class
split is the point of the run.

### Dataset
9 classes, 2,190 images (train 1,575 / val 400 / test 215), **22,957 boxes**.

> The Roboflow export mixed bounding-box rows and polygon rows, and Ultralytics mis-reads such files:
> it decides "is this a segment file?" *per file*, so plain bbox rows sharing a file with a polygon get
> re-read as 2-point polygons. That corrupted 8,047 boxes. The dataset used here has already been
> converted so **every label row is a plain bbox**, and cell 3 asserts it. Do not swap in a fresh
> Roboflow export without re-running `polygons_to_bboxes.py`.

### Where this runs
The notebook detects its environment automatically and adapts paths, batch size and worker count:

- **Local kernel** (VS Code Jupyter extension, RTX 3050) -> reads `D:\ml\data\ssid9_bbox` directly,
  no upload. Select the kernel **`D:\ml\ssid-gpu\Scripts\python.exe`** - it is the only environment
  here with a CUDA build of torch.
- **Colab runtime** -> mounts Drive, unpacks `ssid9_colab.zip`, trains on the cloud T4.

**Run both arms on the same machine.** Batch size is chosen from the available VRAM, so training Run A
on a T4 and Run B locally would compare a batch-16 model against a batch-8 one.

### Before you run
- *Locally*: **close Chrome first.** This machine has a 28.7 GB commit limit against a 4.4 GB pagefile
  on a full C: drive, and Chrome alone holds ~11 GB. Cell 6 checks this and refuses to continue if
  there is not enough headroom - three earlier attempts died with `WinError 1455` and OpenCV
  `Insufficient memory`.
- *Colab*: upload **`ssid9_colab.zip`** (542 MB, at `D:\ml\ssid9_colab.zip`) to the **root of My
  Drive**, and pick a **T4 GPU** runtime.

## 1. Environment

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import platform, torch
print("environment :", "Colab runtime" if IN_COLAB else "local kernel")
print("python      :", platform.python_version())
print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if torch.cuda.is_available():
    print("VRAM (GB)   : %.1f" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

assert torch.cuda.is_available(), (
    "No GPU. In Colab: Runtime > Change runtime type > T4 GPU. "
    "Locally: use the D:\\ml\\ssid-gpu venv, which has the CUDA build of torch."
)

In [ ]:
# Pinned to the version used for every other run in this project, so numbers stay comparable.
if IN_COLAB:
    %pip -q install ultralytics==8.4.114
import ultralytics
print("ultralytics", ultralytics.__version__)

## 2. Data

In [ ]:
import time, zipfile
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP = Path('/content/drive/MyDrive/ssid9_colab.zip')
    assert ZIP.exists(), f"{ZIP} not found - upload ssid9_colab.zip to the root of My Drive."
    # Unpack to local SSD: training directly off the Drive FUSE mount is dramatically slower.
    t0 = time.time()
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
    print(f"unpacked in {time.time() - t0:.0f}s")

    DATA_DIR   = Path('/content/ssid9_bbox')
    RUNS_ROOT  = Path('/content/runs')
    RESULTS    = Path('/content/results')
    REPORT_PY  = Path('/content/report_training_results.py')
else:
    DATA_DIR   = Path(r'D:\ml\data\ssid9_bbox')
    RUNS_ROOT  = Path(r'D:\ml\runs')
    RESULTS    = Path(r'C:\Users\USER\OneDrive\Documents\BEAM\VideoFrameExtractor\01_frame-extractor-tool\results')
    REPORT_PY  = Path(r'C:\Users\USER\OneDrive\Documents\BEAM\VideoFrameExtractor\01_frame-extractor-tool\report_training_results.py')

assert DATA_DIR.is_dir(), f"dataset not found at {DATA_DIR}"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
for split in ('train', 'valid', 'test'):
    print(f"{split:6s} images={len(list((DATA_DIR/split/'images').glob('*'))):5d} "
          f"labels={len(list((DATA_DIR/split/'labels').glob('*.txt'))):5d}")

In [ ]:
# Write a data.yaml with ABSOLUTE paths. Roboflow ships '../train/images', which resolves relative to
# the yaml's own directory and quietly points somewhere wrong; being explicit removes that whole class
# of bug, and the report script needs a resolvable yaml per run anyway.
import collections, yaml

names = yaml.safe_load((DATA_DIR / 'data.yaml').read_text())['names']
DATA_YAML = (DATA_DIR.parent / 'ssid9.yaml')
DATA_YAML.write_text(yaml.dump({
    'train': str(DATA_DIR / 'train' / 'images'),
    'val':   str(DATA_DIR / 'valid' / 'images'),
    'test':  str(DATA_DIR / 'test' / 'images'),
    'nc': len(names),
    'names': names,
}, sort_keys=False))
print(DATA_YAML.read_text())

# Guard against the mixed polygon/bbox corruption ever coming back through a fresh export.
bad = tot = 0
dist = {s: collections.Counter() for s in ('train', 'valid', 'test')}
for s in dist:
    for lf in (DATA_DIR / s / 'labels').glob('*.txt'):
        for line in lf.read_text().splitlines():
            if not line.strip():
                continue
            tot += 1
            parts = line.split()
            if len(parts) != 5:
                bad += 1
            else:
                dist[s][names[int(parts[0])]] += 1

print(f"total rows {tot}, non-bbox rows {bad}")
assert bad == 0, "Mixed polygon/bbox labels detected - run polygons_to_bboxes.py before training."
for s, c in dist.items():
    print(f"\n{s} ({sum(c.values())} boxes)")
    for n, v in c.most_common():
        print(f"   {n:20s} {v}")

## 3. Shared configuration

Both runs read every value from this one cell, so the two arms cannot drift apart.
`NO_AUGMENT` is applied to **Run A only**.

Note `Stitch Scissors` has just **16 train / 10 val / 3 test** instances - report its number, but draw
no conclusion from it. `finger` (7,157 train boxes) dominates every other class, which is why no
class-balancing is applied here: dropping finger-heavy images would throw away most of the dataset.

In [ ]:
EPOCHS = 50
IMGSZ  = 640
SEED   = 42
DEVICE = 0
MODEL  = 'yolo11n.pt'

# BATCH and WORKERS are derived from the hardware ONCE, here, so both arms are guaranteed to get the
# same values. Do not hand-edit them between Run A and Run B: if Ultralytics auto-shrinks the batch
# mid-run because of an OOM, the two arms stop being comparable and the experiment is void.
_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 16 if _vram_gb >= 8 else 8     # a 4 GB laptop card OOMs at 16 with yolo11n @ 640

# Windows + CUDA: every dataloader worker is a new process that re-imports torch and maps the CUDA
# DLLs. On a machine with a small pagefile that reliably dies with "WinError 1455 (paging file is too
# small)", so run the loader in-process there. Colab gives 2 vCPU and has no such problem.
WORKERS = 2 if IN_COLAB else 0

print(f"VRAM {_vram_gb:.1f} GB -> BATCH={BATCH}, WORKERS={WORKERS}")
if not IN_COLAB:
    # This machine has a 28.7 GB commit limit against a 4.4 GB pagefile on a full C: drive, so Windows
    # cannot grow it. Thresholds below are calibrated on OBSERVED failures, not guessed: runs died with
    # "WinError 1455 (paging file too small)" and OpenCV "Insufficient memory" at 3.3-3.6 GB free.
    # 6 GB leaves clear margin above that; 10 GB is comfortable.
    import subprocess
    free_mb = int(subprocess.run(
        ['powershell', '-NoProfile', '-Command',
         '(Get-CimInstance Win32_OperatingSystem).FreeVirtualMemory'],
        capture_output=True, text=True).stdout.strip() or 0) // 1024
    print(f"\nLOCAL RUN - free commit: {free_mb} MB")
    assert free_mb > 6000, (
        f"Only {free_mb} MB of commit free. Runs have died at 3.3-3.6 GB with WinError 1455 / OpenCV "
        f"'Insufficient memory'. Close Chrome and any other heavy apps, then re-run this cell."
    )
    if free_mb < 10000:
        print("  tight but workable with WORKERS=0. If it dies with WinError 1455 or an OpenCV "
              "memory error, close more apps and restart the kernel.")
    else:
        print("  comfortable headroom.")

COMMON = dict(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
              seed=SEED, workers=WORKERS, device=DEVICE, exist_ok=True)

# Exhaustive on purpose: the control arm must stay augmentation-free even if a future Ultralytics
# release turns another knob on by default, and it makes each run's args.yaml self-documenting.
NO_AUGMENT = dict(
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0, bgr=0.0,
    mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, close_mosaic=0,
)

RUN_A = str(RUNS_ROOT / 'ssid9_noaug')   # augmentation OFF
RUN_B = str(RUNS_ROOT / 'ssid9_aug')     # augmentation ON (Ultralytics defaults)
print(COMMON)
print(NO_AUGMENT)

## 4. Run A - augmentation OFF

In [ ]:
import csv, time
from ultralytics import YOLO


def summarize(run, tag):
    # Headline numbers for a finished run, straight from results.csv.
    rows = [{k.strip(): v for k, v in r.items()}
            for r in csv.DictReader(open(f'{run}/train/results.csv'))]
    peak, ep = max((float(r['metrics/mAP50(B)']), int(float(r['epoch']))) for r in rows)
    print(f"{tag}: {len(rows)} epochs in {float(rows[-1]['time'])/60:.1f} min | "
          f"peak mAP50 {peak:.4f} @ epoch {ep} | final mAP50 {float(rows[-1]['metrics/mAP50(B)']):.4f} "
          f"| final mAP50-95 {float(rows[-1]['metrics/mAP50-95(B)']):.4f}")


# Resume guard: re-running this notebook must not silently throw away a finished 45-minute run.
# Delete the run directory to force a retrain.
if (Path(RUN_A) / 'train' / 'weights' / 'best.pt').exists():
    print(f"Run A already complete - reusing {RUN_A}\n(delete that directory to retrain)\n")
    summarize(RUN_A, 'Run A (no-aug)')
else:
    t0 = time.time()
    YOLO(MODEL).train(project=RUN_A, name='train', **COMMON, **NO_AUGMENT)
    print(f"\nRun A finished in {(time.time() - t0) / 60:.1f} min")
    summarize(RUN_A, 'Run A (no-aug)')

In [ ]:
# args.yaml is the RESOLVED config Ultralytics actually trained with - the only trustworthy proof that
# augmentation was really off. Assert rather than eyeball it.
import yaml

AUG_KEYS = ['mosaic', 'fliplr', 'flipud', 'hsv_h', 'hsv_s', 'hsv_v', 'scale', 'translate',
            'degrees', 'shear', 'perspective', 'bgr', 'mixup', 'cutmix', 'copy_paste', 'close_mosaic']

a = yaml.safe_load(open(f'{RUN_A}/train/args.yaml'))
for k in AUG_KEYS:
    print(f"  {k:14s} {a[k]}")
assert all(float(a[k]) == 0 for k in AUG_KEYS), "Run A is NOT augmentation-free - stop and investigate."
print("\nOK - Run A is augmentation-free")

## 5. Run B - augmentation ON (Ultralytics defaults)

In [ ]:
if (Path(RUN_B) / 'train' / 'weights' / 'best.pt').exists():
    print(f"Run B already complete - reusing {RUN_B}\n(delete that directory to retrain)\n")
    summarize(RUN_B, 'Run B (aug)')
else:
    t0 = time.time()
    YOLO(MODEL).train(project=RUN_B, name='train', **COMMON)   # no NO_AUGMENT -> defaults apply
    print(f"\nRun B finished in {(time.time() - t0) / 60:.1f} min")
    summarize(RUN_B, 'Run B (aug)')

In [ ]:
# The experiment is only valid if augmentation is the ONLY difference. Prove it, don't assume it.
b = yaml.safe_load(open(f'{RUN_B}/train/args.yaml'))

print("augmentation (A -> B):")
for k in AUG_KEYS:
    print(f"  {k:14s} {a[k]!s:8s} -> {b[k]}")

CONTROLLED = ['model', 'epochs', 'batch', 'imgsz', 'seed', 'workers', 'device', 'optimizer',
              'lr0', 'lrf', 'momentum', 'weight_decay', 'warmup_epochs', 'patience', 'amp',
              'deterministic']
print("\ncontrolled variables:")
for k in CONTROLLED:
    print(f"  {k:16s} {a.get(k)!s:12s} {b.get(k)}")

mismatch = [k for k in CONTROLLED if str(a.get(k)) != str(b.get(k))]
assert not mismatch, f"These must be identical across arms but differ: {mismatch}"
assert any(float(b[k]) > 0 for k in AUG_KEYS), "Run B has no augmentation - the two arms are identical!"
print("\nOK - augmentation is the only difference between the runs")

## 6. Report

`report_training_results.py` writes every figure as its **own file** and every table as both `.csv` and
`.md`. Classification/regression-only outputs (ROC-AUC one-vs-rest, residual plots, R2) are skipped -
they have no meaning for a detector; the detection equivalents (PR curve per class, F1-confidence,
per-class AP, error-analysis grid) are produced instead.

In [ ]:
import shutil, subprocess, sys

# The report script resolves each run's dataset from <run_dir>/data.yaml (that is where the local
# training script puts it). Training straight from Ultralytics here does not create one, so copy it in
# - without this the script silently skips every validation-derived output: both confusion matrices,
# the PR / F1 curves and the per-class tables.
for run in (RUN_A, RUN_B):
    shutil.copy(DATA_YAML, Path(run) / 'data.yaml')

# Run via the ! magic rather than subprocess.run(capture_output=True): the magic streams the child's
# output straight into this cell, so it is visible live AND saved with the notebook. (An earlier
# subprocess-based version returned stdout=None here even though the script itself ran fine.)
!"{sys.executable}" "{REPORT_PY}" --runs noaug="{RUN_A}" aug="{RUN_B}" --out "{RESULTS}" --split val --device {DEVICE}

## 7. Results

In [ ]:
from IPython.display import Markdown, display

for name in ['benchmark_comparison', 'hyperparameter_summary',
             'noaug_per_class_metrics', 'aug_per_class_metrics', 'class_distribution']:
    p = RESULTS / f'{name}.md'
    if p.exists():
        display(Markdown(p.read_text(encoding='utf-8')))
    else:
        print(f"(missing: {name}.md)")

In [ ]:
# Every figure inline, so a saved copy of this notebook carries the entire report with it.
from IPython.display import Image, display

figs = sorted(RESULTS.glob('*.png'))
print(f"{len(figs)} figures\n")
for f in figs:
    display(Markdown(f"**{f.name}**"))
    display(Image(filename=str(f)))

## 8. Save the artifacts

In [ ]:
import shutil

if IN_COLAB:
    OUT = Path('/content/drive/MyDrive/ssid9_results')
else:
    OUT = Path(r'D:\ml\ssid9_results')
OUT.mkdir(parents=True, exist_ok=True)

shutil.copytree(RESULTS, OUT / 'results', dirs_exist_ok=True)
for tag, run in (('noaug', RUN_A), ('aug', RUN_B)):
    dst = OUT / tag
    dst.mkdir(exist_ok=True)
    shutil.copy(f'{run}/train/weights/best.pt', dst / 'best.pt')    # the model itself
    shutil.copy(f'{run}/train/results.csv',     dst / 'results.csv')  # per-epoch log
    shutil.copy(f'{run}/train/args.yaml',       dst / 'args.yaml')    # proof of configuration

print("saved to", OUT)
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1e6:.1f} MB)")

---
### Save this notebook with its outputs

**File -> Save a copy in Drive**, or **File -> Download -> Download .ipynb**. Either keeps every table
and figure above embedded in the file, so it opens as a complete report.

### What to send back for analysis
- `results/benchmark_comparison.md`
- `results/noaug_per_class_metrics.md` and `results/aug_per_class_metrics.md`
- both `args.yaml` files
- the wall-clock minutes printed by each training cell